# charts_project.ipynb

## Проект "Дашборд конверсий"

**Цель:** построение графиков визитов и регистраций, агрегация по платформам и периодам рекламных кампаний.

**Входные данные:**
- API: визиты, регистрации
- Параметры: API_URL, DATE_BEGIN, DATE_END из .env
- CSV: ads.csv

**Выходные данные:**
- Результат обработки данных, файлы: conversion.json, ads.json
- Графики (линейные, столбчатые, с накоплением) в папке `charts`

In [26]:
import os

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from dotenv import load_dotenv
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from pathlib import Path

### Блок: запросы API и определение переменных

In [27]:
load_dotenv()

api_url = os.getenv("API_URL")
date_begin = os.getenv("DATE_BEGIN")
date_end = os.getenv("DATE_END")

url_visits = api_url + "/visits"
url_reg = api_url + "/registrations"
params = {"begin": date_begin, "end": date_end}

date_start = pd.to_datetime(date_begin).normalize()
date_end = pd.to_datetime(date_end).normalize()

output_dir = Path("charts")
output_dir.mkdir(parents=True, exist_ok=True)  


### Блок: загрузка данных

In [ ]:
response_visits = requests.get(url_visits, params=params, timeout=10)

response_reg = requests.get(url_reg, params=params, timeout=10)

data_visits = response_visits.json()

data_reg = response_reg.json()

df_visits = pd.DataFrame(data_visits)

df_reg = pd.DataFrame(data_reg)

df_ads = pd.read_csv("./ads.csv")

df_visits["datetime"] = pd.to_datetime(df_visits["datetime"], errors="coerce")
df_visits = df_visits.rename(columns={"datetime": "date"})

df_reg["datetime"] = pd.to_datetime(df_reg["datetime"], errors="coerce")
df_reg = df_reg.rename(columns={"datetime": "date"})

df_ads["date"] = pd.to_datetime(df_ads["date"], errors="coerce")


### Блок: расчет метрик

In [ ]:
# Обработка визитов: последний визит на visit_id
df_visits_unique = (
    df_visits.sort_values("date")
    .drop_duplicates(subset="visit_id", keep="last")
    .copy()
)

# группируем по дням, обнуляя время, но сохраняя тип datetime
df_visits_unique["date_group"] = df_visits_unique["date"].dt.normalize()

visits_agg = (
    df_visits_unique.groupby(["date_group", "platform"], observed=True)
    .size()
    .reset_index(name="visits")
)

# Агрегация регистраций по дням и платформе
df_reg["date_group"] = df_reg["date"].dt.normalize()

reg_agg = (
    df_reg.groupby(["date_group", "platform"], observed=True)
    .size()
    .reset_index(name="registrations")
)

# Объединение и конверсия
df_merged = visits_agg.merge(
    reg_agg, on=["date_group", "platform"], how="left"
)
df_merged["registrations"] = df_merged["registrations"].fillna(0).astype(int)

df_merged["conversion"] = df_merged["registrations"] / df_merged["visits"]
df_merged["conversion"] = (df_merged["conversion"] * 100).fillna(0)

df_merged["date_group"] = pd.to_datetime(df_merged["date_group"])

#df_merged["date_group"] = (
#    df_merged["date_group"].astype("datetime64[ns]").astype("int64") // 1_000_000
#)

df_merged["conversion"] = df_merged["conversion"].round(6)

df_merged = df_merged.reset_index(drop=True)

df_merged = df_merged.sort_values(["date_group", "platform"]).reset_index(drop=True)

# Сохранение в JSON (timestamp-даты, orient=columns)
df_merged.to_json("./conversion.json", orient="columns", force_ascii=False)


### Блок: дообогащение данных из таблицы ads

In [ ]:
# Подготовка рекламы (агрегация по дате)
df_ads['date_group'] = df_ads['date'].dt.normalize()

ads_agg = (
    df_ads.groupby("date_group", observed=True)
    .agg(
        cost=("cost", "sum"),
        utm_campaign=(
            "utm_campaign",
            lambda x: x.iloc[0] if len(x) > 0 else "none",
        ),
    )
    .reset_index()
)

ads_agg = ads_agg.sort_values("date_group")

# Объединение с конверсиями 
df_final = df_merged.merge(ads_agg, on="date_group", how="left")

# Приводим дату к Unix‑мс (только сейчас!)
#df_final["date_group"] = pd.to_datetime(df_final["date_group"])
#df_final["date_group"] = df_final["date_group"].astype("int64") // 1_000_000

# Сортируем и сбрасываем индекс для предсказуемого JSON
df_final = df_final.sort_values(["date_group", "platform"]).reset_index(drop=True)

# Сохранение в JSON
df_final.to_json("./ads.json", orient="columns", force_ascii=False)


### Визуализация
- [Итоговые визиты]()
- [Итоговые визиты с разбивкой по платформам]()
- [Итоговые регистрации]()
- [Итоговые регистрации с разбивкой по платформе]()
- [Итоговые конверсии]()
- [Конверсия по каждой платформе]()
- [Стоимости реклам]()
- [Визиты и регистрации с выделением рекламных кампаний]()

### Итоговые визиты

In [ ]:
df = df_final.copy()

# Группируем по дате и суммируем визиты
visits_by_date_df = (
    df
    .groupby("date_group", as_index=False)["visits"]
    .sum()
)

# Строим график
plt.figure(figsize=(14, 6))

x_labels = visits_by_date_df["date_group"].dt.strftime("%Y-%m-%d")
x_pos = range(len(x_labels))

# Добавляем подписи над столбцами
bars = plt.bar(
    x_pos,      # ось X: даты из колонки
    visits_by_date_df["visits"],          # высота: визиты из колонки
    color="#93c9df"
)

plt.bar_label(bars, padding=3, fontsize=9, color="black", fmt="%d")

ax = plt.gca()
ax.grid(axis="y", linestyle="--", alpha=0.3)
ax.grid(axis="x", visible=False)

# Настройка оси X (даты)
ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
plt.xticks(rotation=45, ha="right")

plt.title("Total Visits")
plt.xlabel("Date")
plt.ylabel("Visits")
plt.tight_layout()

output_dir = Path("charts")
output_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(output_dir / "total_visits.png", dpi=300)
plt.show()

### Итоговые визиты с разбивкой по платформам

In [ ]:
# Настройка стилей
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 7)
plt.rcParams["font.size"] = 12

# Фильтрация платформ 
allowed_platforms = ["web", "android", "ios"]
df_plot = df_final[df_final["platform"].isin(allowed_platforms)].copy()

# Агрегация: суммируем визиты по дате и платформе
visits_pivot = df_plot.pivot_table(
    index="date_group", columns="platform", values="visits", aggfunc="sum"
)

# Сортируем индекс (даты)
visits_pivot = visits_pivot.sort_index()

# Построение столбчатой диаграммы
ax = visits_pivot.plot(
    kind="bar",
    stacked=True,
    figsize=(14, 7),
    width=0.8,
    edgecolor="black",
    linewidth=0.5,
)

# Настраиваем ось X
ax.set_xticklabels(
    [ts.strftime("%Y-%m-%d") for ts in visits_pivot.index],
    rotation=45,
    ha="right",
)

plt.title(
    "Итоговые визиты с разбивкой по платформам (web, android, ios)",
    fontsize=14,
)
plt.xlabel("Дата", fontsize=12)
plt.ylabel("Визиты", fontsize=12)
plt.legend(title="Платформа", loc="upper right")
plt.tight_layout()

# Сохраняем график
plt.savefig(output_dir / "visits_by_platform_bar.png", dpi=300)
plt.show()


## Итоговые регистрации по дням

In [ ]:
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 7)
plt.rcParams["font.size"] = 12


# Группируем по дате и суммируем регистрации
reg_by_date = df_final.groupby("date_group")["registrations"].sum()

# Строим график
plt.figure(figsize=(14, 6))
bars = plt.bar(reg_by_date.index, reg_by_date.values, color="#93c9df")

# Добавляем подписи над столбцами
plt.bar_label(bars, padding=3, fontsize=9, color="black", fmt="%d")

# Убираем вертикальные сетки
ax = plt.gca()
ax.grid(
    axis="y", linestyle="--", alpha=0.3
)  # оставляем только горизонтальные (опционально)
ax.grid(axis="x", visible=False)  # отключаем вертикальные

# Настраиваем ось X: каждый день, без пропусков, наклон 45°
ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
plt.xticks(
    rotation=45, ha="right"
)  # ha='right' — чтобы текст не «улетал» за край

plt.title("Регистрации по дням")
plt.xlabel("Дата")
plt.ylabel("Регистрации")
# plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig(output_dir / "total_day_registration.png", dpi=300)
plt.show()


## Итоговые регистрации по неделям

In [ ]:
# Подготовка данных
df = df_final.copy()

# Считаем номер недели 
days_from_start = (df["date_group"] - date_start).dt.days
df["week_num"] = days_from_start // 7

# Агрегируем по номеру недели
reg_by_week = (
    df
    .groupby("week_num")["registrations"]
    .sum()
    .reset_index(name="registrations")
)

# Формируем подписи: "неделя 0", "неделя 1", ...
reg_by_week["week_label"] = "неделя " + reg_by_week["week_num"].astype(str)

# График
plt.figure(figsize=(14, 6))

bars = plt.bar(
    range(len(reg_by_week)),          # позиции: 0, 1, 2, ...
    reg_by_week["registrations"],      # высоты столбцов
    color="#93c9df",
    width=0.7                          # ширина столбцов (чтобы не слипались)
)

plt.bar_label(bars, padding=3, fontsize=9, color="black", fmt="%d")

ax = plt.gca()
ax.grid(axis="y", linestyle="--", alpha=0.3)
ax.grid(axis="x", visible=False)

# Подписи оси X — наши кастомные метки
ax.set_xticks(range(len(reg_by_week)))
ax.set_xticklabels(reg_by_week["week_label"], rotation=45, ha="right")

plt.title("Регистрации за неделю")
plt.xlabel("Недели")
plt.ylabel("Регистрации")
plt.tight_layout()

output_dir = Path("charts")
output_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(output_dir / "total_weekly_registration.png", dpi=300)
plt.show()

## Итоговые регистрации с разбивкой по платформе

In [ ]:
# Подготовка данных
df = df_final.copy()

# --- Агрегация: одна строка = одна дата, колонки = платформы ---
reg_by_platform = (
    df
    .groupby(["date_group", "platform"])["registrations"]
    .sum()
    .unstack(fill_value=0)  # платформы → колонки, пропуски = 0
)

# Фиксируем порядок колонок и убеждаемся, что все нужные платформы есть
df_plot = reg_by_platform[allowed_platforms]

# Построение stacked bar 
ax = df_plot.plot(
    kind="bar",
    stacked=True,
    figsize=(14, 6),
    width=0.8
)

# Форматируем метки дат на оси X (index у df_plot — это даты)
tick_labels = [ts.strftime("%Y-%m-%d") for ts in df_plot.index]
ax.set_xticks(range(len(tick_labels)))
ax.set_xticklabels(tick_labels, rotation=45, ha="right")

plt.title("Регистрации по платформам")
plt.xlabel("Дата")
plt.ylabel("Регистрации")

# Легенда справа
plt.legend(title="Платформа", loc="upper right")

plt.tight_layout()  # чтобы легенда и подписи не обрезались

output_dir = Path("charts")
output_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(output_dir / "registrations_by_platform.png", dpi=300)
plt.show()

### Итоговые конверсии

In [ ]:
# Подготовка данных
df = df_final.copy()

# Исключаем платформу 'bot'
df_no_bot = df[df["platform"] != "bot"].copy()

# Агрегируем по дате
df_plot = df_no_bot.groupby("date_group", as_index=False)[
    ["visits", "registrations"]
].sum()

# Итоговая конверсия
df_plot["conversion_total"] = (
    df_plot["registrations"] / df_plot["visits"]
) * 100
df_plot["conversion_total"] = df_plot["conversion_total"].round(2)

# Строим график и добавляем подписи
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(
    df_plot["date_group"],
    df_plot["conversion_total"],
    marker="o",
    linewidth=2,
    color="#2E86AB",
)

# Подписи над точками
for i, row in df_plot.iterrows():
    x_val = row["date_group"]
    y_val = row["conversion_total"]

    # Пропускаем битые значения
    if pd.isna(x_val) or pd.isna(y_val) or np.isinf(y_val):
        continue

    ax.annotate(
        f"{y_val:.1f}%",
        xy=(x_val, y_val),  # координаты точки
        xytext=(0, 8),  # сдвиг вверх на 8 пунктов
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=9,
        color="#2E86AB",
        weight="bold",
    )

ax.set_title("Итоговые конверсии по дням (без bot)", fontsize=14)
ax.set_xlabel("Дата", fontsize=12)
ax.set_ylabel("Конверсия, %", fontsize=12)
ax.grid(True, linestyle="--", alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(output_dir / "total_conversion.png", dpi=300)
plt.show()


### Конверсия по каждой платформе

In [ ]:
# Подготовка данных
df = df_final.copy()

# Убираем bot, если есть
df = df[df["platform"] != "bot"].copy()

# Выбираем нужные платформы
platforms = ["android", "ios", "web"]
df_plot = df.copy()

# Цвета и маркеры
colors = {"android": "#2E86AB", "ios": "#A23B72", "web": "#F18F01"}
markers = {"android": "o", "ios": "s", "web": "^"}

# --- Создаём фигуру с 3 подграфиками (3 строки, 1 колонка) ---
fig, axes = plt.subplots(
    3, 1, figsize=(12, 14), sharex=True
)  # sharex=True — общая ось X (даты)

for ax, platform in zip(axes, platforms):
    sub = df_plot[df_plot["platform"] == platform].sort_values("date_group")

    if sub.empty:
        ax.text(
            0.5,
            0.5,
            f"Нет данных: {platform}",
            transform=ax.transAxes,
            ha="center",
            va="center",
            fontsize=12,
        )
        ax.set_title(f"Конверсия: {platform.upper()}", fontsize=14)
        continue

    # Рисуем линию
    ax.plot(
        sub["date_group"],
        sub["conversion"],
        marker=markers[platform],
        linewidth=2,
        color=colors[platform],
        label=platform.capitalize(),
        markersize=6,
    )

    # Подписи процентов над точками
    for i, row in sub.iterrows():
        x_val = row["date_group"]
        y_val = row["conversion"]

        if pd.isna(x_val) or pd.isna(y_val) or np.isinf(y_val):
            continue

        ax.annotate(
            f"{y_val:.1f}%",
            xy=(x_val, y_val),
            xytext=(0, 8),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=9,
            color=colors[platform],
            weight="bold",
        )

    ax.set_title(f"Конверсия: {platform.upper()}", fontsize=14)
    ax.legend(loc="best")
    ax.grid(True, linestyle="--", alpha=0.3)

# Настройка общей оси X
plt.xticks(rotation=45)
fig.text(
    0.04, 0.5, "Конверсия, %", va="center", rotation="vertical", fontsize=12
)  # общая подпись оси Y
fig.suptitle(
    "Конверсии по платформам (android, ios, web)", fontsize=16, y=0.995
)
plt.tight_layout()
fig.savefig(
    output_dir / "conversions_by_platform.png", dpi=300, bbox_inches="tight"
)
plt.show()


## Стоимости реклам

In [ ]:
# Подготовка данных
df = df_final.copy()

# Убираем bot
df = df[df["platform"] != "bot"].copy()

# 4. Агрегируем стоимость по дате (суммируем cost по всем платформам за день)
cost_by_date = (
    df_plot.groupby("date_group", as_index=False)["cost"]
    .sum()
    .sort_values("date_group")
)

# Построение графика
plt.figure(figsize=(12, 6))

plt.plot(
    cost_by_date["date_group"],
    cost_by_date["cost"],
    color="#2E86AB",
    linewidth=2,
    marker="o",
    markersize=6,
)

# Добавляем подписи стоимости над точками
for i, row in cost_by_date.iterrows():
    x_val = row["date_group"]
    y_val = row["cost"]

    if pd.isna(x_val) or pd.isna(y_val) or np.isinf(y_val):
        continue
    if y_val == 0:
        label = f"{y_val:.0f}"
    else:
        label = f"{y_val:.0f} RUB"
    plt.annotate(
        label,  # без дробной части (стоимость обычно целая)
        xy=(x_val, y_val),
        xytext=(0, 8),  # сдвиг вверх на 8 пунктов
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=9,
        color="black",
        # weight='bold'
    )

plt.title("Стоимость рекламы по дням", fontsize=14)
plt.xlabel("Дата", fontsize=12)
plt.ylabel("Стоимость", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(output_dir / "ad_cost.png", dpi=300, bbox_inches="tight")
plt.show()


## Визиты и регистрации с выделением рекламных кампаний

In [ ]:
# Подготовка данных
df = df_final.copy()

# Убираем bot
df = df[df["platform"] != "bot"].copy()

# Агрегация по дням 
daily = df_plot.groupby("date_group", as_index=False).agg(
    visits_unique=("visits", "sum"), users_unique=("registrations", "sum")
)

mean_visits = daily["visits_unique"].mean()
mean_users = daily["users_unique"].mean()

# Периоды рекламных кампаний (utm_campaign не пустой и не None) 
campaign_periods = []
for camp, sub in df_plot.groupby("utm_campaign", dropna=True):
    start = sub["date_group"].min()
    end = sub["date_group"].max()
    campaign_periods.append((camp, start, end))

# Яркие цвета для кампаний
bright_colors = [
    "#FF5733",  # оранжевый
    "#33FF57",  # зелёный
    "#3357FF",  # синий
    "#F333FF",  # пурпурный
    "#FFFF33",  # жёлтый
]
camp_color_map = {
    camp: bright_colors[i % len(bright_colors)]
    for i, (camp, _, _) in enumerate(campaign_periods)
}

#  Построение двух графиков в одном окне 
fig, axes = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

# График 1: Визиты 
ax1 = axes[0]

line_visits = ax1.plot(
    daily["date_group"],
    daily["visits_unique"],
    color="#2E86AB",
    linewidth=2,
    marker="o",
    markersize=4,
    label="Визиты (уникальные)",
)[0]

ax1.axhline(
    mean_visits,
    color="red",
    linestyle="--",
    linewidth=1.5,
    label=f"Среднее: {mean_visits:.1f}",
)

# Фон только для периодов с кампаниями
for camp, start, end in campaign_periods:
    color = camp_color_map[camp]
    ax1.axvspan(start, end, color=color, alpha=0.25, label=camp)

ax1.set_title(
    "Уникальные визиты по дням с выделением рекламных кампаний", fontsize=14
)
ax1.set_ylabel("Визиты", fontsize=12)
ax1.grid(True, linestyle=":", alpha=0.4)

# Легенда 1 (своя)
handles1 = [line_visits]
handles1.append(
    Line2D(
        [0],
        [0],
        color="red",
        linestyle="--",
        lw=1.5,
        label=f"Среднее: {mean_visits:.1f}",
    )
)

for camp, _, _ in campaign_periods:
    color = camp_color_map[camp]
    handles1.append(
        Patch(facecolor=color, edgecolor="none", alpha=0.25, label=camp)
    )

ax1.legend(handles=handles1, loc="upper left", fontsize=9, frameon=True)


# График 2: Пользователи 
ax2 = axes[1]

line_users = ax2.plot(
    daily["date_group"],
    daily["users_unique"],
    color="#A23B72",
    linewidth=2,
    marker="s",
    markersize=4,
    label="Пользователи (уникальные)",
)[0]

ax2.axhline(
    mean_users,
    color="red",
    linestyle="--",
    linewidth=1.5,
    label=f"Среднее: {mean_users:.1f}",
)

# Фон кампаний (без label, чтобы не дублировать в легенде)
for camp, start, end in campaign_periods:
    color = camp_color_map[camp]
    ax2.axvspan(start, end, color=color, alpha=0.25)

ax2.set_title(
    "Уникальные пользователи по дням с выделением рекламных кампаний",
    fontsize=14,
)
ax2.set_xlabel("Дата", fontsize=12)
ax2.set_ylabel("Пользователи", fontsize=12)
ax2.grid(True, linestyle=":", alpha=0.4)

# Легенда 2 (своя): линия, среднее, и по одному образцу каждой кампании
handles2 = [line_users]
handles2.append(
    Line2D(
        [0],
        [0],
        color="red",
        linestyle="--",
        lw=1.5,
        label=f"Среднее: {mean_users:.1f}",
    )
)

for camp, _, _ in campaign_periods:
    color = camp_color_map[camp]
    handles2.append(
        Patch(facecolor=color, edgecolor="none", alpha=0.25, label=camp)
    )

ax2.legend(handles=handles2, loc="upper left", fontsize=9, frameon=True)

plt.xticks(rotation=45)
plt.tight_layout()
fig.savefig(
    output_dir / "visits_users_campaigns_clean.png", dpi=300, bbox_inches="tight"
)
plt.show()
